# POC — CEDA Iztapalapa: DENUE establishments inside the Central de Abasto geofence

Proof of concept for the `ceda_pipeline`:

1. Load the official **DENUE** bulk extract for CDMX (estado 09, published 2026-05).
2. Take the **Central de Abasto de la Ciudad de México** site polygon from OpenStreetMap (way `1199350930`).
3. Filter DENUE establishments inside the geofence (`point_in_geofence`, polygon mode — boundary counts as inside).
4. Enrich with the fixed **SCIAN 2023** catalog and map the result.

In [1]:
import sys, json
from pathlib import Path

import pandas as pd
import geopandas as gpd
import folium

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

from ceda_pipeline.data_sources import load_scian_rows
from ceda_pipeline.scian import load_scian_catalog, map_scian_codes
from ceda_pipeline.geocode import build_geofence, point_in_geofence
from ceda_pipeline.denue import normalize_denue_rows

DENUE_CSV = ROOT / "data/denue/conjunto_de_datos/denue_inegi_09_.csv"
CEDA_GEOJSON = ROOT / "data/ceda_iztapalapa_osm.geojson"
SCIAN_XLSX = ROOT / "data/scian/scian_2023_categorias_y_productos.xlsx"
OUT_DIR = ROOT / "data/outputs"
OUT_DIR.mkdir(exist_ok=True)

## 1. CEDA geofence

OSM way **1199350930** — *Central de Abasto de la Ciudad de México*, Iztapalapa (market DF-F7 in the 2012 directory).

In [2]:
ceda = gpd.read_file(CEDA_GEOJSON)
ceda[["name", "osm_id", "lat", "lon", "display_name"]]

,name,osm_id,lat,lon,display_name
0,Central de Abasto de la Ciudad de México,1199350930,19.373122,-99.090529,"Central de Abasto de la Ciudad de México, Mexi..."


In [3]:
# polygon vertices as (lat, lon); also compute its bbox for a fast prefilter
poly_latlon = [(y, x) for x, y in ceda.geometry.iloc[0].exterior.coords]
geofence = build_geofence({"type": "polygon", "coordinates": poly_latlon})
minx, miny, maxx, maxy = ceda.total_bounds
print(f"polygon vertices: {len(poly_latlon)}")
print(f"bbox lat [{miny:.4f}, {maxy:.4f}]  lon [{minx:.4f}, {maxx:.4f}]")

polygon vertices: 133
bbox lat [19.3670, 19.3793]  lon [-99.0996, -99.0817]


## 2. Load DENUE CDMX + spatial prefilter

In [4]:
cols = ["id","nom_estab","codigo_act","nombre_act","latitud","longitud",
        "tipo_vial","nom_vial","numero_ext","letra_ext","tipo_asent","nomb_asent",
        "cod_postal","cve_mun","municipio","entidad","per_ocu","fecha_alta",
        "tipoCenCom","nom_CenCom","num_local"]
denue = pd.read_csv(DENUE_CSV, usecols=cols, dtype={"cve_mun": str}, encoding="latin-1")
denue["latitud"] = pd.to_numeric(denue["latitud"], errors="coerce")
denue["longitud"] = pd.to_numeric(denue["longitud"], errors="coerce")
print(len(denue), "establishments in CDMX")

# cheap bbox prefilter before the exact polygon test
near = denue[
    denue["latitud"].between(miny - 0.01, maxy + 0.01)
    & denue["longitud"].between(minx - 0.01, maxx + 0.01)
].copy()
print(len(near), "inside buffered bbox")

462732 establishments in CDMX
16019 inside buffered bbox


In [5]:
near["inside"] = [point_in_geofence(la, lo, geofence)
                  for la, lo in zip(near["latitud"], near["longitud"])]
inside = near[near["inside"]]
print(f"{len(inside)} establishments inside the CEDA polygon")

7278 establishments inside the CEDA polygon


## 3. SCIAN enrichment (SCIAN 2023, fixed titles)

In [6]:
scian_rows = load_scian_rows(str(SCIAN_XLSX))
catalog = load_scian_catalog(scian_rows, version="2023", version_date="2023-07-19")
inside = inside.copy()
inside["codigo_act"] = inside["codigo_act"].astype(str).str.strip()
mapped, unknown = map_scian_codes(inside["codigo_act"].tolist(), catalog)
by_code = {m["code"]: m for m in mapped}
inside["scian_title"] = inside["codigo_act"].map(lambda c: by_code.get(c, {}).get("title"))

def sector_of(code):
    p = int(str(code)[:2])
    for m in ("31-33", "48-49"):
        lo, hi = [int(x) for x in m.split("-")]
        if lo <= p <= hi:
            return m
    return str(code)[:2]

inside["sector"] = inside["codigo_act"].map(sector_of)
inside["sector_title"] = inside["sector"].map(lambda s: catalog[s].title if s in catalog else s)
print("unmapped codes:", sorted(set(unknown)))
inside[["nom_estab","codigo_act","scian_title","sector_title"]].head()

unmapped codes: []


,nom_estab,codigo_act,scian_title,sector_title
19,FRUTAS CASA GRANDES,115113,Beneficio de productos agrícolas,"Agricultura, cría y explotación de animales, a..."
215,CENTRO DE ATENCION A CLIENTES MAGDALENA ID 150,221123,Comercialización de energía eléctrica,"Generación, transmisión, distribución y comerc..."
537,POZO DE AGUA SAN LORENZO TEZONCO 2,221312,"Captación, tratamiento y suministro de agua re...","Generación, transmisión, distribución y comerc..."
1415,GRUPO EFEM,236111,Edificación de vivienda unifamiliar,Construcción
2448,ABASTECEDORA DE LONAS HERLINDA,314912,Confección de productos de textiles recubierto...,Industrias manufactureras


## 4. Establishments by SCIAN sector

In [7]:
sector_counts = (
    inside.groupby(["sector", "sector_title"], as_index=False)
    .size().sort_values("size", ascending=False)
)
sector_counts

,sector,sector_title,size
5,46,Comercio al por menor,5010
4,43,Comercio al por mayor,1562
15,72,Servicios de alojamiento temporal y de prepara...,431
16,81,Otros servicios excepto actividades gubernamen...,111
9,53,Servicios inmobiliarios y de alquiler de biene...,43
3,31-33,Industrias manufactureras,34
6,48-49,"Transportes, correos y almacenamiento",15
17,93,"Actividades legislativas, gubernamentales, de ...",15
8,52,Servicios financieros y de seguros,14
11,56,Servicios de apoyo a los negocios y manejo de ...,12


In [8]:
inside.groupby(["codigo_act","scian_title"], as_index=False).size() \
      .sort_values("size", ascending=False).head(15)

,codigo_act,scian_title,size
58,461130,Comercio al por menor de frutas y verduras fre...,3484
23,431130,Comercio al por mayor de frutas y verduras fre...,953
88,466312,Comercio al por menor de plantas y flores natu...,478
57,461123,Comercio al por menor de pescados y mariscos,203
54,461110,"Comercio al por menor en tiendas de abarrotes,...",166
59,461140,Comercio al por menor de semillas y granos ali...,144
22,431123,Comercio al por mayor de pescados y mariscos,120
149,722518,Restaurantes que preparan otro tipo de aliment...,111
19,431110,Comercio al por mayor de abarrotes,110
25,431150,Comercio al por mayor de semillas y granos ali...,98


## 5. Map

In [9]:
import math

# DENUE assigns ONE coordinate per nave/section of the complex: the 7,278
# matched rows collapse to ~433 unique points (each a section of a nave).
# Render one circle per unique coordinate, sized by establishment count.
groups = inside.groupby(["latitud", "longitud"]).agg(
    n=("id", "size"),
    cencom=("nom_CenCom", lambda s: s.fillna("").str.strip().replace("", "—").mode().iloc[0]),
    acts=("scian_title", lambda s: s.value_counts().head(3).to_dict()),
    locals_sample=("num_local", lambda s: s.dropna().astype(str).head(6).tolist()),
).reset_index()
print(len(groups), "unique geocode points")

palette = ["#e41a1c","#377eb8","#4daf4a","#984ea3","#ff7f00","#a65628",
           "#f781bf","#999999","#66c2a5","#fc8d62","#8da0cb","#e78ac3","#a6d854","#ffd92f"]
sectors = sector_counts["sector"].tolist()
color_of = {s: palette[i % len(palette)] for i, s in enumerate(sectors)}
dom_sector = inside.groupby(["latitud","longitud"])["sector"].agg(lambda s: s.mode().iloc[0])
groups = groups.merge(dom_sector.rename("dom_sector"), on=["latitud","longitud"])

m = folium.Map(location=[float(ceda.lat.iloc[0]), float(ceda.lon.iloc[0])], zoom_start=15, tiles="OpenStreetMap")
folium.GeoJson(
    json.loads(ceda.to_json())["features"][0]["geometry"],
    name="CEDA boundary", style_function=lambda _: {"color": "#333", "weight": 2, "fillOpacity": 0.08},
).add_to(m)

for _, r in groups.iterrows():
    acts_html = "<br>".join(f"{k}: {v}" for k, v in r["acts"].items())
    folium.CircleMarker(
        [r["latitud"], r["longitud"]],
        radius=4 + math.sqrt(r["n"]) * 1.2,
        color=color_of.get(r["dom_sector"], "#000"), fill=True, fill_opacity=0.65, weight=1,
        popup=(f"<b>{r['n']} establishments</b> at this point<br>"
               f"Complex: {r['cencom']}<br>{acts_html}<br>"
               f"Locales: {', '.join(r['locals_sample'])}"),
        tooltip=f"{r['n']} negocios — {r['cencom']}",
    ).add_to(m)
m

433 unique geocode points


## 6. Export matched rows

In [10]:
out = inside.drop(columns=["inside"])
out.to_csv(OUT_DIR / "ceda_iztapalapa_denue.csv", index=False)
gpd.GeoDataFrame(out, geometry=gpd.points_from_xy(out.longitud, out.latitud), crs="EPSG:4326") \
   .to_file(OUT_DIR / "ceda_iztapalapa_denue.geojson", driver="GeoJSON")
m.save(OUT_DIR / "ceda_iztapalapa_map.html")
print(f"wrote {len(out)} rows + map to {OUT_DIR}")

wrote 7278 rows + map to /home/chonkydev/Documents/Github Repos/CEDA-in-Central-Markets/data/outputs


## Notes / next steps

- DENUE vintages: the bulk extract is the *current* directory (metadata in `data/denue/metadatos/`), not year-sliced — the pipeline's `year` contract still needs per-year DENUE snapshots.
- The strict pipeline path (`run_pipeline`) can reuse these inputs; `normalize_denue_rows` now accepts raw DENUE columns (`nom_estab`, `codigo_act`, `latitud`, `longitud`) and composes `address` from address components.
- Geocoding the market address is *not* needed here — the OSM polygon is the geofence.